# 03 — Customer Feature Engineering & RFM Segmentation
**Project:** CustomerPulse AI — E-commerce Customer Intelligence & Churn Prediction System

### Objective
Inspect the 17 behavioral customer features, compare interpretable RFM rule-based segments against log-scaled K-Means clusters (`k=4`), and evaluate revenue concentration across customer tiers.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CUSTOMER_FEATURES_PATH, CUSTOMER_SEGMENTS_PATH

df_feat = pd.read_csv(CUSTOMER_FEATURES_PATH, dtype={"customer_id": str})
df_seg = pd.read_csv(CUSTOMER_SEGMENTS_PATH, dtype={"customer_id": str, "rfm_score_str": str})
df = pd.merge(df_feat, df_seg, on="customer_id", how="inner")
print(f"Loaded {len(df):,} segmented customers.")
df.head()

## 1. Rule-Based RFM Segment Summary

In [ ]:
rfm_summary = (
    df.groupby("rfm_segment")
    .agg(
        customer_count=("customer_id", "count"),
        mean_recency_days=("recency_days", "mean"),
        mean_frequency_orders=("frequency_orders", "mean"),
        mean_monetary_gross=("monetary_gross", "mean"),
        total_segment_revenue=("monetary_gross", "sum"),
    )
    .round(2)
    .sort_values("total_segment_revenue", ascending=False)
)
rfm_summary["revenue_share_pct"] = ((rfm_summary["total_segment_revenue"] / rfm_summary["total_segment_revenue"].sum()) * 100).round(2)
rfm_summary

## 2. Unsupervised K-Means (`k=4`) Cluster Summary & Cross-Tabulation

In [ ]:
kmeans_summary = (
    df.groupby("kmeans_segment_label")
    .agg(
        customer_count=("customer_id", "count"),
        mean_recency_days=("recency_days", "mean"),
        mean_frequency_orders=("frequency_orders", "mean"),
        mean_monetary_gross=("monetary_gross", "mean"),
        total_cluster_revenue=("monetary_gross", "sum"),
    )
    .round(2)
    .sort_values("total_cluster_revenue", ascending=False)
)
display(kmeans_summary)

pd.crosstab(df["rfm_segment"], df["kmeans_segment_label"], margins=True)